# ExoSignal Observatory - NASA DR24 Deep Learning Run

Visible Colab runbook for the new dataset path. This version uses the public Google Research AstroNet precomputed NASA Kepler DR24 global/local light-curve views so training can finish in Colab without downloading about 90 GB of raw Kepler FITS files.

In [ ]:
# 1) Runtime + GPU check
import os, sys, json, platform, shutil, subprocess, textwrap
from pathlib import Path
print('Python', sys.version)
print('Platform', platform.platform())
try:
    import tensorflow as tf
    print('TensorFlow before install', tf.__version__)
    print('GPUs', tf.config.list_physical_devices('GPU'))
except Exception as exc:
    print('TensorFlow not imported yet:', exc)
print('Disk before cleanup:')
!df -h /content
!du -h -d 1 /content 2>/dev/null | sort -h | tail -20

In [ ]:
# 2) Hard cleanup of Colab temporary storage
from pathlib import Path
import shutil, os, subprocess
paths = [
    '/content/exosignal-observatory',
    '/content/exosignal-bootstrap',
    '/content/exosignal_nasa_dr24_final_artifacts.zip',
    '/content/exosignal_astronet_dr24_final_artifacts.zip',
    '/content/exosignal_nasa_dr24_training_executed.ipynb',
    '/content/dr24_build.log',
    '/content/run_dr24.ipynb',
    '/content/kepler',
    '/content/astronet',
    '/content/lightcurves',
    '/content/.cache',
    '/root/.cache/pip',
    '/root/.cache/huggingface',
]
for item in paths:
    p = Path(item)
    if p.is_dir():
        shutil.rmtree(p, ignore_errors=True)
    elif p.exists():
        try:
            p.unlink()
        except OSError:
            pass
print('Cleaned project/cache paths. Colab system image usage cannot be deleted by notebook code.')
!df -h /content
!du -h -d 1 /content 2>/dev/null | sort -h | tail -20

In [ ]:
# 3) Clone latest GitHub repo
REPO_URL = 'https://github.com/Maaskk/exosignal-observatory.git'
PROJECT_DIR = '/content/exosignal-observatory'
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git log --oneline -3
!ls -la | sed -n '1,80p'

In [ ]:
# 4) Install dependencies
!pip -q install -r requirements-deep.txt
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('No GPU detected. Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# 5) Run configuration
DATASET_PROFILE = 'astronet-dr24'
SEQUENCE_LENGTH = 1024
TRAIN_EPOCHS = 40
BATCH_SIZE = 32
ENSEMBLE = 3
TTA_RUNS = 8
BACKBONE = 'inceptiontime'
CALIBRATION = 'isotonic'
print(json.dumps({
    'dataset_profile': DATASET_PROFILE,
    'sequence_length': SEQUENCE_LENGTH,
    'epochs': TRAIN_EPOCHS,
    'batch_size': BATCH_SIZE,
    'ensemble': ENSEMBLE,
    'tta_runs': TTA_RUNS,
    'backbone': BACKBONE,
    'calibration': CALIBRATION,
}, indent=2))

In [ ]:
# 6) Download + convert the new NASA Kepler DR24 AstroNet dataset
# Source: Google Research AstroNet precomputed DR24 global/local light-curve views.
# This avoids raw Kepler FITS bulk download, which AstroNet documents as about 90 GB for the DR24 training subset.
!python -u scripts/build_astronet_dr24_dataset.py --download --clean

In [ ]:
# 7) Data distribution after cleaning/conversion
import json
from pathlib import Path
import pandas as pd
manifest = json.loads(Path('data/astronet_dr24/manifest.json').read_text())
print(json.dumps(manifest, indent=2)[:6000])
for split in ['train', 'val', 'test']:
    path = Path(f'data/astronet_dr24/processed/{split}.parquet')
    df = pd.read_parquet(path)
    print('
===', split.upper(), '===')
    print('rows:', len(df))
    print('class distribution:', df['disposition'].value_counts().to_dict())
    print('unique stars:', df['kepid'].nunique() if 'kepid' in df else 'n/a')
    print('median period:', float(pd.to_numeric(df['period_days'], errors='coerce').median()))
    print('median duration hrs:', float(pd.to_numeric(df['duration_hrs'], errors='coerce').median()))
    print('file MB:', round(path.stat().st_size / 1024 / 1024, 2))
    sample = df.iloc[0]
    print('sample global bytes:', len(sample['flux_global']), 'local bytes:', len(sample['flux_local']))

In [ ]:
# 8) Memory plan for 512 / 1024 / 2048 sequence lengths
!python scripts/train_deep_learning.py --dataset-profile astronet-dr24 --print-memory-plan --sweep-sequence-lengths 512,1024,2048

In [ ]:
# 9) Train the deep model on the new AstroNet DR24 dataset
!python -u scripts/train_deep_learning.py   --dataset-profile astronet-dr24   --task binary   --model-family hybrid   --backbone {BACKBONE}   --sequence-length {SEQUENCE_LENGTH}   --epochs {TRAIN_EPOCHS}   --batch-size {BATCH_SIZE}   --mixed-precision   --augment   --loss focal   --lr-schedule cosine   --ensemble {ENSEMBLE}   --calibration {CALIBRATION}   --tta-runs {TTA_RUNS}

In [ ]:
# 10) Final metrics table
import json
from pathlib import Path
metrics_path = Path('models/deep_model_metrics.json')
config_path = Path('models/deep_model_config.json')
metrics = json.loads(metrics_path.read_text())
config = json.loads(config_path.read_text())
keys = ['dataset_profile', 'backbone', 'training_rows', 'validation_rows', 'test_rows', 'sequence_length', 'ensemble_members', 'threshold', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
for key in keys:
    print(f'{key}:', metrics.get(key, config.get(key)))
print('confusion_matrix:', json.dumps(metrics.get('confusion_matrix'), indent=2))

In [ ]:
# 11) Export trained artifacts and reports
from pathlib import Path
import zipfile
zip_path = Path('/content/exosignal_astronet_dr24_final_artifacts.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pattern in ['models/*.json', 'models/*.keras', 'models/experiments/**/*.keras', 'reports/*.json', 'reports/*.md', 'data/astronet_dr24/manifest.json']:
        for file in Path('.').glob(pattern):
            zf.write(file, file.as_posix())
print('artifact zip:', zip_path, round(zip_path.stat().st_size / 1024 / 1024, 2), 'MB')
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print('download skipped:', exc)

## Scientific caution

The model prioritizes exoplanet candidates. It does not officially confirm a planet; confirmation needs expert vetting and follow-up observations.